In [33]:
import pandas as pd

# Load csv files

orders = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_customers_dataset.csv')
items = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_sellers_dataset.csv')
categories = pd.read_csv('../data/raw/olist_trimmed_1500_relational/product_category_name_translation.csv')
geolocation = pd.read_csv('../data/raw/olist_trimmed_1500_relational/olist_geolocation_dataset.csv')

In [34]:
tables = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "categories": categories,
    "geolocation": geolocation,
}

expected_counts = {
    "orders": 1208,
    "customers": 1208,
    "items": 1500,
    "payments": 1303,
    "reviews": 1298,
    "products": 1096,
    "sellers": 494,
    "categories": 57,
    "geolocation": 956,
}

row_check = []

for name, df in tables.items():
    row_check.append({
        "table": name,
        "actual_rows": len(df),
        "expected_rows": expected_counts[name],
        "passed": len(df) == expected_counts[name]
    })

row_check = pd.DataFrame(row_check)
display(row_check)

,table,actual_rows,expected_rows,passed
0,orders,1208,1208,True
1,customers,1208,1208,True
2,items,1500,1500,True
3,payments,1303,1303,True
4,reviews,1298,1298,True
5,products,1096,1096,True
6,sellers,494,494,True
7,categories,57,57,True
8,geolocation,956,956,True


In [35]:
# 2. Convert identifiers and ZIP prefixes to strings

string_columns = {
    "orders": ["order_id", "customer_id"],
    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix"
    ],
    "items": ["order_id", "product_id", "seller_id"],
    "payments": ["order_id"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id", "seller_zip_code_prefix"],
    "categories": ["product_category_name"],
    "geolocation": ["geolocation_zip_code_prefix"],
}

for table_name, columns in string_columns.items():
    df = tables[table_name]

    for column in columns:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()     # converts every value in your table's column into text( .astype("string") ), trims extra spaces( .str.strip() )

In [36]:
# 3. Parse date columns

date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "items": ["shipping_limit_date"],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

date_conversion_results = []

for table_name, columns in date_columns.items():
    df = tables[table_name]

    for column in columns:
        original_non_null = df[column].notna().sum()
        df[column] = pd.to_datetime(df[column], errors="coerce")
        converted_non_null = df[column].notna().sum()

        date_conversion_results.append({
            "table": table_name,
            "column": column,
            "invalid_dates": original_non_null - converted_non_null
        })

date_conversion_results = pd.DataFrame(date_conversion_results)
display(date_conversion_results)

,table,column,invalid_dates
0,orders,order_purchase_timestamp,0
1,orders,order_approved_at,0
2,orders,order_delivered_carrier_date,0
3,orders,order_delivered_customer_date,0
4,orders,order_estimated_delivery_date,0
5,items,shipping_limit_date,0
6,reviews,review_creation_date,0
7,reviews,review_answer_timestamp,0


In [37]:
# 4. Check date ranges

print("Purchase date range:")
print(orders["order_purchase_timestamp"].min())
print(orders["order_purchase_timestamp"].max())

print("\nPurchase months:")
purchase_months = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M") # takes exact dates and times and simplifies them into just the Year and Month (like turning 2026-11-21 13:51:00 into 2026-11)
    .sort_values()
    .astype(str)
    .unique()   
)

print(purchase_months)
print("Number of observed months:", len(purchase_months))

Purchase date range:
2016-10-03 09:44:50
2018-08-28 10:57:13

Purchase months:
<StringArray>
['2016-10', '2016-12', '2017-01', '2017-02', '2017-03', '2017-04', '2017-05',
 '2017-06', '2017-07', '2017-08', '2017-09', '2017-10', '2017-11', '2017-12',
 '2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07',
 '2018-08']
Length: 22, dtype: str
Number of observed months: 22


In [38]:
# 5. Check invalid date order

invalid_date_order = orders[
    (orders["order_approved_at"] < orders["order_purchase_timestamp"]) |
    (orders["order_delivered_carrier_date"] < orders["order_approved_at"]) |
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
]

display(invalid_date_order)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
169,c76b75e541b06461ec9cf363fd962ee3,90df085f8a512665dce9ae9e923b0b83,delivered,2018-05-23 15:50:05,2018-05-23 16:16:01,2018-05-23 14:51:00,2018-06-07 19:31:51,2018-07-02
188,c0cb48c270009fc128318b2f9eda3bec,246295516c97fc54aa274c6fd2222602,delivered,2018-04-22 17:47:10,2018-04-24 18:34:06,2018-04-23 18:02:33,2018-05-04 18:17:35,2018-05-15
245,7dda251cd0ccf3b0e0ff56d62b779fdc,d506c0d4abd2528e9fddb923894a9dda,delivered,2017-05-21 08:00:14,2017-05-22 10:41:43,2017-05-22 10:40:10,2017-05-23 08:49:42,2017-06-02
358,2e55db958a20257bc57183a4e74e6b47,91db678752ee943f3565ef444a7072ad,delivered,2018-07-30 16:04:45,2018-08-01 16:31:33,2018-07-31 10:01:00,2018-08-03 15:08:38,2018-08-29
361,73696c4354b75af323cf793aa474acfc,c99b3cf4d3a8c1829fc7b4c71bd3e43c,delivered,2017-05-05 07:48:46,2017-05-10 21:32:36,2017-05-10 09:47:50,2017-05-16 06:50:09,2017-05-26
528,b866af202be0692766081310cd4085e1,d1800078046ed2e5ae1b0792b695c56e,delivered,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17
572,9cc8a145d8be894ab07cdbd8503c4887,a3fc5aa6679a45c8d0dd3036c9ee246b,delivered,2017-11-28 10:44:35,2017-12-03 23:36:19,2017-11-28 22:15:58,2017-12-18 19:44:08,2017-12-21


In [39]:
# 6. Check order statuses

status_summary = (
    orders["order_status"]
    .value_counts(dropna=False) # returns a Series (not a DataFrame)
    .rename_axis("order_status")  #  Gives a name to the unnamed index of the Series.
    .reset_index(name="count") # Converts the Series into a DataFrame and creates a fresh numeric index (0, 1, 2...).

)

display(status_summary)

,order_status,count
0,delivered,1208


In [40]:
# 7. Check primary-key duplicates

primary_key_checks = {
    "orders.order_id": orders["order_id"].duplicated(keep=False),
    "customers.customer_id": customers["customer_id"].duplicated(keep=False),
    "products.product_id": products["product_id"].duplicated(keep=False),
    "sellers.seller_id": sellers["seller_id"].duplicated(keep=False),
}

for key_name, duplicate_mask in primary_key_checks.items():
    print(key_name, "duplicates:", duplicate_mask.sum())

orders.order_id duplicates: 0
customers.customer_id duplicates: 0
products.product_id duplicates: 0
sellers.seller_id duplicates: 0


In [41]:
# 8. Check composite keys

duplicate_item_keys = items[
    items.duplicated(
        subset=["order_id", "order_item_id"],
        keep=False
    )
]

duplicate_payment_keys = payments[
    payments.duplicated(
        subset=["order_id", "payment_sequential"],
        keep=False
    )
]

display(duplicate_item_keys)
display(duplicate_payment_keys)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [42]:
# 9. Check duplicate review IDs

duplicate_review_ids = reviews[
    reviews.duplicated("review_id", keep=False)
].sort_values("review_id")

print("Repeated review_id rows:", len(duplicate_review_ids))
display(duplicate_review_ids)

Repeated review_id rows: 254


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
280,0467560f511c516ddaa54a60edb0c291,0c995611a99f81268d859184a416f1db,5,NaN,NaN,2017-02-09,2017-02-14 15:58:44
323,0467560f511c516ddaa54a60edb0c291,55b2e390d5d80ada31ad1b795ebeb087,5,NaN,NaN,2017-02-09,2017-02-14 15:58:44
309,0655af56f10bc3ef8e3810610828f294,84f5e6c0a0e3155e38c00f434ba90ce8,5,NaN,Muito rápido a entrega,2017-05-18,2017-05-19 19:15:22
650,0655af56f10bc3ef8e3810610828f294,d9e44c3fd2ce16086619f299e92e12d8,5,NaN,Muito rápido a entrega,2017-05-18,2017-05-19 19:15:22
796,090aa2980b0aea78cd1ae73447d7d55b,66f943fd426ac8716ef6b6b97ace8bd8,4,NaN,NaN,2018-05-10,2018-05-11 10:52:52
...,...,...,...,...,...,...,...
1183,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21,2017-07-26 13:45:15
1048,fd53ace363f9af2bc44ae028b2e926d6,c66985d27ca5899e6058749d3432daf4,5,NaN,NaN,2017-08-16,2017-08-19 10:34:40
475,fd53ace363f9af2bc44ae028b2e926d6,b6f999c6eccbf4df9af77d67c77a8e52,5,NaN,NaN,2017-08-16,2017-08-19 10:34:40
17,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07


In [43]:
# 10. Check missing values

missing_summary = []

for name, df in tables.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_summary.append({
                "table": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(df) * 100,
                    2
                )
            })

missing_summary = pd.DataFrame(missing_summary)
display(missing_summary)

,table,column,missing_count,missing_percent
0,orders,order_approved_at,1,0.08
1,reviews,review_comment_title,1200,92.45
2,reviews,review_comment_message,774,59.63
3,products,product_category_name,28,2.55
4,products,product_name_lenght,28,2.55
5,products,product_description_lenght,28,2.55
6,products,product_photos_qty,28,2.55


In [44]:
# 11. Check order-to-customer relationships

missing_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

display(missing_customers)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [45]:
# 12. Check item relationships

missing_item_orders = items[
    ~items["order_id"].isin(orders["order_id"])
]

missing_item_products = items[
    ~items["product_id"].isin(products["product_id"])
]

missing_item_sellers = items[
    ~items["seller_id"].isin(sellers["seller_id"])
]

display(missing_item_orders)
display(missing_item_products)
display(missing_item_sellers)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [46]:
# 13. Check payment and review relationships

missing_payment_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

missing_review_orders = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

display(missing_payment_orders)
display(missing_review_orders)

,order_id,payment_sequential,payment_type,payment_installments,payment_value


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [47]:
# 14. Check that every order has items, payments, and reviews

orders_without_items = orders[
    ~orders["order_id"].isin(items["order_id"])
]

orders_without_payments = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

orders_without_reviews = orders[
    ~orders["order_id"].isin(reviews["order_id"])
]

print("Orders without items:", len(orders_without_items))
print("Orders without payments:", len(orders_without_payments))
print("Orders without reviews:", len(orders_without_reviews))

Orders without items: 0
Orders without payments: 0
Orders without reviews: 0


In [48]:
# 15. Check customer identity

customer_identity = pd.DataFrame({
    "metric": [
        "customer records",
        "unique people",
        "repeat customers"
    ],
    "value": [
        customers["customer_id"].nunique(),
        customers["customer_unique_id"].nunique(),
        (
            customers.groupby("customer_unique_id")
            .size()
            .gt(1)
            .sum()
        )
    ]
})

display(customer_identity)

,metric,value
0,customer records,1208
1,unique people,539
2,repeat customers,380


In [49]:
# 16. Validate review scores

invalid_review_scores = reviews[
    ~reviews["review_score"].between(1, 5)
]

display(invalid_review_scores)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [50]:
# 17. Validate numeric fields

invalid_items = items[
    (items["price"] < 0) |
    (items["freight_value"] < 0)
]

invalid_payments = payments[
    (payments["payment_value"] < 0) |
    (payments["payment_installments"] < 1)
]

display(invalid_items)
display(invalid_payments)
print()
zero_payment_values = payments[
    payments["payment_value"] == 0
]

zero_freight_values = items[
    items["freight_value"] == 0
]

print("Zero payment values:", len(zero_payment_values))
print("Zero freight values:", len(zero_freight_values))

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


,order_id,payment_sequential,payment_type,payment_installments,payment_value



Zero payment values: 1
Zero freight values: 10


In [51]:
# 18. Check ZIP-prefix relationships

missing_customer_geolocation = customers[
    ~customers["customer_zip_code_prefix"].isin(
        geolocation["geolocation_zip_code_prefix"]
    )
]

missing_seller_geolocation = sellers[
    ~sellers["seller_zip_code_prefix"].isin(
        geolocation["geolocation_zip_code_prefix"]
    )
]

display(missing_customer_geolocation)
display(missing_seller_geolocation)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


,seller_id,seller_zip_code_prefix,seller_city,seller_state


In [52]:
# 19. Check category translations

missing_category_translations = products[
    products["product_category_name"].notna() &
    ~products["product_category_name"].isin(
        categories["product_category_name"]
    )
]

display(missing_category_translations)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [53]:
# 20. Keep item and payment calculations separate

item_totals = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_revenue=("price", "sum"),
        freight_collected=("freight_value", "sum")
    )
)

payment_totals = (
    payments.groupby("order_id", as_index=False)
    .agg(
        order_payment_total=("payment_value", "sum")
    )
)

order_level_data = (
    orders
    .merge(item_totals, on="order_id", how="left")
    .merge(payment_totals, on="order_id", how="left")
)

display(order_level_data.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_revenue,freight_collected,order_payment_total
0,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04,132.4,14.05,146.45
1,a685d016c8a26f71a0bb67821070e398,911e4c37f5cafe1604fe6767034bf1ae,delivered,2017-03-13 18:14:36,2017-03-13 18:14:36,2017-03-22 14:03:09,2017-04-06 13:37:16,2017-03-30,84.9,14.36,99.26
2,f175d67589e059cbbda956f10f0702e6,a4555d257db29ef5f838cec4101c5208,delivered,2017-01-05 13:34:35,2017-01-07 03:45:40,2017-01-11 16:25:57,2017-01-17 15:52:28,2017-02-13,9.9,8.72,18.62
3,7e3e92dc3eaf5389f32da97228f7dd0e,62d50e9a714ac024c196d9548594e2aa,delivered,2017-07-02 20:47:24,2017-07-02 21:03:55,2017-07-03 11:42:08,2017-07-11 20:22:31,2017-07-25,79.9,15.31,95.21
4,dc400373e624e9b0435847a79813c658,4a00f00d951c167739888a1f16c56e8e,delivered,2018-01-14 18:49:14,2018-01-14 18:59:30,2018-01-16 20:15:54,2018-01-20 14:28:43,2018-01-30,39.9,9.34,49.24


In [54]:
# 21. Create an exception report

from pathlib import Path

# Paths relative to Python/pre-analysis.ipynb
quality_dir = Path("../data/quality")
quality_dir.mkdir(parents=True, exist_ok=True)

exceptions = []


def add_exceptions(df, table_name, reason):
    if not df.empty:
        result = df.copy()
        result.insert(0, "table_name", table_name)
        result.insert(1, "exception_reason", reason)
        exceptions.append(result)


add_exceptions(
    invalid_date_order,
    "orders",
    "Invalid date sequence"
)

add_exceptions(
    duplicate_item_keys,
    "order_items",
    "Duplicate composite item key"
)

add_exceptions(
    duplicate_payment_keys,
    "payments",
    "Duplicate composite payment key"
)

add_exceptions(
    invalid_review_scores,
    "reviews",
    "Invalid review score"
)

add_exceptions(
    invalid_items,
    "order_items",
    "Invalid item numeric value"
)

add_exceptions(
    invalid_payments,
    "payments",
    "Invalid payment numeric value"
)


if exceptions:
    exception_report = pd.concat(
        exceptions,
        ignore_index=True
    )
else:
    exception_report = pd.DataFrame(
        columns=["table_name", "exception_reason"]
    )


exception_file = quality_dir / "rejected_rows.csv"

exception_report.to_csv(
    exception_file,
    index=False
)

print(f"Exception report saved to: {exception_file}")
print(f"Total exception rows: {len(exception_report)}")

display(exception_report)

PermissionError: [Errno 13] Permission denied: '..\\data\\quality\\rejected_rows.csv'

In [ ]:
# # 22. Save prepared files

# prepared_dir = Path("../data/prepared")
# prepared_dir.mkdir(parents=True, exist_ok=True)

# prepared_files = {
#     "orders_prepared.csv": orders,
#     "customers_prepared.csv": customers,
#     "order_items_prepared.csv": items,
#     "payments_prepared.csv": payments,
#     "reviews_prepared.csv": reviews,
#     "products_prepared.csv": products,
#     "sellers_prepared.csv": sellers,
#     "category_translation_prepared.csv": categories,
#     "geolocation_prepared.csv": geolocation,
# }

# for filename, df in prepared_files.items():
#     output_path = prepared_dir / filename
#     df.to_csv(output_path, index=False)
#     print(f"Saved: {output_path}")